# Atelier Préparation de Données Images

## Contexte

Une entreprise souhaite développer un système d'intelligence artificielle capable de reconnaître automatiquement le type de déchet présent sur une photographie afin d'améliorer le tri des déchets. Le modèle devra classer chaque image dans l'une des catégories suivantes :

- **cardboard** : cartons ondulés, cartons plats...
- **plastic** : bouteilles, emballages plastiques...
- **paper** : feuilles, journaux...
- **glass** : bouteilles et objets en verre...
- **metal** : canettes, boîtes métalliques...
- **trash** : emballages bonbons, tasses jetables...

Le problème est que les images collectées proviennent de plusieurs sources. Elles ne sont donc pas homogènes : dimensions différentes, formats différents, images RGB et grayscale, certaines images sont trop petites, certaines images sont corrompues, quelques images sont vides, images dupliquées, quelques images placées dans le mauvais dossier, classes déséquilibrées.

L'objectif de l'atelier est donc de construire un jeu de données images propre et homogène, prêt à être utilisé pour entraîner un modèle de Machine Learning ou de Deep Learning.

# Partie 1 – Exploration du dataset

In [1]:
# Partie 1 - Étape 1 : Imports nécessaires
import os
import numpy as np
import pandas as pd
from PIL import Image

In [2]:
# Partie 1 - Étape 2 : Fonction d'extraction des métadonnées d'une image
def extraire_metadonnees(chemin_image, classe):
    """
    Extrait les métadonnées d'une image : nom, classe, format, mode,
    largeur, hauteur, écart-type des pixels, nombre de canaux, taille.
    Retourne un dictionnaire, avec corrompue=True si l'image ne peut pas être lue.
    """
    nom_fichier = os.path.basename(chemin_image)
    taille_octets = os.path.getsize(chemin_image)

    try:
        with Image.open(chemin_image) as img:
            img.verify()  # vérifie l'intégrité du fichier sans le charger complètement

        # Réouverture nécessaire après verify() (qui "consomme" l'objet image)
        with Image.open(chemin_image) as img:
            largeur, hauteur = img.size
            format_img = img.format
            mode_img = img.mode

            img_array = np.array(img)
            ecart_type = img_array.std()
            nb_canaux = 1 if img_array.ndim == 2 else img_array.shape[2]

        return {
            'nom_fichier': nom_fichier,
            'classe': classe,
            'format': format_img,
            'mode': mode_img,
            'largeur': largeur,
            'hauteur': hauteur,
            'ecart_type_pixels': ecart_type,
            'nb_canaux': nb_canaux,
            'taille_octets': taille_octets,
            'corrompue': False
        }

    except Exception as e:
        return {
            'nom_fichier': nom_fichier,
            'classe': classe,
            'format': None,
            'mode': None,
            'largeur': None,
            'hauteur': None,
            'ecart_type_pixels': None,
            'nb_canaux': None,
            'taille_octets': taille_octets,
            'corrompue': True
        }

In [3]:
# Partie 1 - Étape 3 : Parcours de tout le dataset et construction du tableau récapitulatif
dossier_raw = "../data/raw"
classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

resultats = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        metadonnees = extraire_metadonnees(chemin_complet, classe)
        resultats.append(metadonnees)

df_images = pd.DataFrame(resultats)
df_images.shape

(1032, 10)

In [4]:
# Partie 1 - Étape 4 : Aperçu du résultat
print(df_images.head())
print("\nNombre d'images par classe :")
print(df_images['classe'].value_counts())
print("\nNombre d'images corrompues :", df_images['corrompue'].sum())

        nom_fichier     classe format mode  largeur  hauteur  \
0    cardboard1.jpg  cardboard   JPEG  RGB    512.0    384.0   
1   cardboard10.jpg  cardboard   JPEG  RGB    512.0    384.0   
2  cardboard100.jpg  cardboard   JPEG  RGB    512.0    384.0   
3  cardboard101.jpg  cardboard   JPEG  RGB    512.0    384.0   
4  cardboard102.jpg  cardboard   JPEG  RGB    512.0    384.0   

   ecart_type_pixels  nb_canaux  taille_octets  corrompue  
0          40.586504        3.0          17333      False  
1          42.577273        3.0          21683      False  
2          46.121684        3.0          14884      False  
3          72.264255        3.0          14289      False  
4          48.389753        3.0          18015      False  

Nombre d'images par classe :
classe
paper        252
plastic      224
glass        188
cardboard    169
metal        149
trash         50
Name: count, dtype: int64

Nombre d'images corrompues : 6


# Partie 2 – Détecter les images corrompues

In [5]:
# Partie 2 : Fonction de détection d'image corrompue
def est_corrompue(chemin_image):
    """
    Vérifie si une image est corrompue (fichier illisible, tronqué,
    ou n'étant pas une image valide). Retourne True si corrompue, False sinon.
    """
    try:
        with Image.open(chemin_image) as img:
            img.verify()
        return False
    except Exception:
        return True

In [6]:
# Partie 2 : Application de la fonction sur tout le dataset
images_corrompues = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        if est_corrompue(chemin_complet):
            images_corrompues.append({'nom_fichier': nom_fichier, 'classe': classe})

df_corrompues = pd.DataFrame(images_corrompues)
print("Nombre d'images corrompues détectées :", len(df_corrompues))
df_corrompues

Nombre d'images corrompues détectées : 6


,nom_fichier,classe
0,cardboard83.jpg,cardboard
1,glass74.jpg,glass
2,metal48.jpg,metal
3,paper213.jpg,paper
4,plastic13.jpg,plastic
5,trash3.jpg,trash


# Partie 3 – Détecter les images vides

In [7]:
# Partie 3 : Fonction de détection d'image vide
def est_vide(chemin_image, seuil_ecart_type=5):
    """
    Vérifie si une image est "vide" : entièrement noire, entièrement blanche,
    ou avec très peu de variation de pixels (écart-type sous le seuil).
    Retourne True si l'image est considérée comme vide, False sinon.
    Retourne None si l'image est corrompue (ne peut pas être analysée).
    """
    if est_corrompue(chemin_image):
        return None

    with Image.open(chemin_image) as img:
        img_array = np.array(img)

    ecart_type = img_array.std()
    moyenne = img_array.mean()

    # Image entièrement noire (moyenne proche de 0) ou blanche (moyenne proche de 255)
    entierement_noire = moyenne < 5
    entierement_blanche = moyenne > 250
    peu_de_variation = ecart_type < seuil_ecart_type

    return entierement_noire or entierement_blanche or peu_de_variation

In [8]:
# Partie 3 : Application de la fonction sur tout le dataset
images_vides = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        resultat = est_vide(chemin_complet)
        if resultat:
            images_vides.append({'nom_fichier': nom_fichier, 'classe': classe})

df_vides = pd.DataFrame(images_vides)
print("Nombre d'images vides détectées :", len(df_vides))
df_vides

Nombre d'images vides détectées : 2


,nom_fichier,classe
0,image-blanche-512x384.jpg,cardboard
1,image-blanche-512x384.jpg,metal


# Partie 4 – Détecter les différences de résolution

In [9]:
# Partie 4 - Question 1 : Statistiques de résolution
# On exclut les images corrompues, qui n'ont pas de largeur/hauteur valides
df_valides = df_images[df_images['corrompue'] == False].copy()

print("Résolution minimale (largeur) :", df_valides['largeur'].min())
print("Résolution minimale (hauteur) :", df_valides['hauteur'].min())
print("Résolution maximale (largeur) :", df_valides['largeur'].max())
print("Résolution maximale (hauteur) :", df_valides['hauteur'].max())

print("\nRésolutions les plus fréquentes :")
resolutions = df_valides.groupby(['largeur', 'hauteur']).size().sort_values(ascending=False)
print(resolutions.head(10))

Résolution minimale (largeur) : 32.0
Résolution minimale (hauteur) : 32.0
Résolution maximale (largeur) : 512.0
Résolution maximale (hauteur) : 384.0

Résolutions les plus fréquentes :
largeur  hauteur
512.0    384.0      1013
32.0     32.0          5
40.0     40.0          4
48.0     32.0          4
dtype: int64


In [11]:
# Partie 4 - Question 2 : Images ne respectant pas la contrainte minimale de 64x64
images_trop_petites = df_valides[(df_valides['largeur'] < 64) | (df_valides['hauteur'] < 64)]

print("Nombre d'images trop petites :", len(images_trop_petites))
images_trop_petites[['nom_fichier', 'classe', 'largeur', 'hauteur']]

Nombre d'images trop petites : 13


,nom_fichier,classe,largeur,hauteur
20,cardboard117.jpg,cardboard,48.0,32.0
77,cardboard22.jpg,cardboard,32.0,32.0
133,cardboard70.jpg,cardboard,40.0,40.0
171,glass100.jpg,glass,40.0,40.0
229,glass15.jpg,glass,48.0,32.0
268,glass21.jpg,glass,32.0,32.0
270,glass23.jpg,glass,32.0,32.0
383,metal121.jpg,metal,48.0,32.0
413,metal2.jpg,metal,32.0,32.0
421,metal26.jpg,metal,40.0,40.0


# Partie 5 – Détecter les différents canaux

In [12]:
# Partie 5 : Nombre d'images selon leur nombre de canaux
df_valides['nb_canaux'].value_counts()

nb_canaux
3.0    1006
4.0      18
1.0       2
Name: count, dtype: int64

# Partie 6 – Détecter les doublons

In [14]:
# Partie 6 : Fonction de calcul du hash d'une image
import hashlib

def calculer_hash(chemin_image):
    """
    Calcule un hash MD5 du contenu binaire d'une image.
    Retourne None si l'image est corrompue.
    """
    if est_corrompue(chemin_image):
        return None

    with open(chemin_image, 'rb') as f:
        contenu = f.read()
    return hashlib.md5(contenu).hexdigest()

In [15]:
# Partie 6 : Application - détection des doublons sur tout le dataset
hashes = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in os.listdir(dossier_classe):
        chemin_complet = os.path.join(dossier_classe, nom_fichier)
        h = calculer_hash(chemin_complet)
        hashes.append({'nom_fichier': nom_fichier, 'classe': classe, 'hash': h})

df_hashes = pd.DataFrame(hashes)

# Identifier les doublons : hash présent plus d'une fois (en excluant les None des corrompues)
df_hashes_valides = df_hashes[df_hashes['hash'].notna()]
doublons = df_hashes_valides[df_hashes_valides.duplicated(subset='hash', keep=False)].sort_values('hash')

print("Nombre d'images dupliquées (occurrences totales) :", len(doublons))
doublons

Nombre d'images dupliquées (occurrences totales) : 30


,nom_fichier,classe,hash
732,paper77ty.jpg,paper,098ca18d235424eec30cc10f743ff155
731,paper77.jpg,paper,098ca18d235424eec30cc10f743ff155
388,metal125po.jpg,metal,0b0dafc63da482605bac609d6e327a0a
387,metal125.jpg,metal,0b0dafc63da482605bac609d6e327a0a
838,plastic171.jpg,plastic,0fde5dfaea544255c189f9a262854200
839,plastic171az.jpg,plastic,0fde5dfaea544255c189f9a262854200
355,image-noire-512x384.png,glass,213b77974c84c6db40cc487743d92981
358,image-noire-512x384.png,metal,213b77974c84c6db40cc487743d92981
167,image-blanche-512x384.jpg,cardboard,2624c04611d93c7ef187d7fd0ab41216
357,image-blanche-512x384.jpg,metal,2624c04611d93c7ef187d7fd0ab41216
